# 📱 K-Means Clustering: Mobile Money User Segmentation in East Africa

**Author:** Jok Akech Atem Mabior | CMU Africa

**Dataset:** Simulated mobile money transaction data inspired by M-Pesa (Kenya) and MTN MoMo (Uganda/Rwanda) platforms.

---

This notebook applies K-Means and Hierarchical clustering to segment mobile money users into behavioral groups, enabling targeted product design and financial inclusion strategies.


## Background: Mobile Money in East Africa

### Market Context

East Africa leads the world in mobile money adoption:

- **M-Pesa (Kenya)**: 35M+ active users, processes $314B+ annually. Launched 2007, now embedded in healthcare, agriculture, and education payments.
- **MTN MoMo (Uganda & Rwanda)**: 14M+ users across the region. Dominant in countries where M-Pesa has limited reach.
- **Airtel Money, Equitel**: Additional platforms serving niche segments.

### Financial Inclusion Impact

**60% of Kenyan adults** use mobile money as their primary financial tool — far exceeding traditional banking penetration. In rural areas, mobile money is often the **only** formal financial service accessible.

### Why User Segmentation Matters

Segmenting users by transaction behavior enables:
1. **Product personalization**: Offer savings products to casual users, credit lines to heavy transactors
2. **Churn prevention**: Identify dormant users before they fully disengage
3. **Revenue optimization**: Design fee structures that maximize volume without alienating price-sensitive users
4. **Risk management**: Identify unusual transaction patterns for fraud detection
5. **Regulatory compliance**: KYC tiers based on transaction volume and frequency


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
from scipy.cluster.hierarchy import dendrogram, linkage
import warnings
warnings.filterwarnings('ignore')
np.random.seed(42)
plt.style.use('seaborn-v0_8')
print("Libraries loaded!")

## Data Generation: 2,000 Mobile Money Users

We simulate 4 behavioral segments:
- **Heavy/Business** (400): High-frequency, high-value transactions; merchant payments
- **Regular** (700): Moderate usage; typical wage earners and small traders
- **Light/Savers** (600): Infrequent transactions; primarily deposit/savings behavior
- **Dormant** (300): Rarely transact; accounts nearly inactive


In [ ]:
np.random.seed(42)

def gen_seg(n, txn_mu, txn_sd, amt_mu, age_mu,
            merchant_prob, intl_mu, days_mu, savings_prob, topup_mu):
    txn = np.random.normal(txn_mu, txn_sd, n).clip(0, 300)
    return pd.DataFrame({
        'monthly_transactions': txn,
        'avg_transaction_ksh': np.random.lognormal(np.log(max(amt_mu,1)), 0.6, n).clip(50, 150000),
        'account_age_months': np.random.normal(age_mu, age_mu*0.25, n).clip(1, 120),
        'merchant_payment_ratio': np.random.beta(max(merchant_prob*8,0.1), max((1-merchant_prob)*8,0.1), n),
        'intl_transfers_pm': np.random.poisson(intl_mu, n).astype(float),
        'days_since_last_txn': np.random.exponential(days_mu, n).clip(0, 365),
        'monthly_topup_count': np.random.poisson(topup_mu, n).clip(0, 60).astype(float),
        'savings_ratio': np.random.beta(max(savings_prob*6,0.1), max((1-savings_prob)*6+0.5,0.1), n),
        'bill_payments_pm': np.random.poisson(txn_mu * 0.08, n).clip(0, 20).astype(float),
        'unique_recipients_pm': np.random.poisson(txn_mu * 0.3, n).clip(0, 80).astype(float),
    })

seg_A = gen_seg(400, 80, 18, 8000, 36, 0.65, 2.5, 2, 0.15, 12)   # Heavy/Business
seg_B = gen_seg(700, 22, 8,  1800, 24, 0.35, 0.4, 7, 0.30, 4)    # Regular
seg_C = gen_seg(600, 6,  3,  400,  18, 0.10, 0.1, 25, 0.60, 1)   # Light/Savers
seg_D = gen_seg(300, 1,  1,  200,  48, 0.05, 0.0, 120, 0.08, 0)  # Dormant

seg_A['true_segment'] = 'Heavy/Business'
seg_B['true_segment'] = 'Regular'
seg_C['true_segment'] = 'Light/Savers'
seg_D['true_segment'] = 'Dormant'

df = pd.concat([seg_A, seg_B, seg_C, seg_D], ignore_index=True)
df = df.sample(frac=1, random_state=42).reset_index(drop=True)
df['monthly_volume_ksh'] = df['monthly_transactions'] * df['avg_transaction_ksh']

print(f"Dataset shape: {df.shape}")
print(f"True segment distribution:\n{df['true_segment'].value_counts()}")
df.describe().round(2)

## Exploratory Data Analysis

### EDA 1: Feature Distribution Overview


In [ ]:
plot_cols = ['monthly_transactions', 'avg_transaction_ksh', 'account_age_months',
             'days_since_last_txn', 'savings_ratio', 'monthly_volume_ksh']
seg_palette = {'Heavy/Business': '#e74c3c', 'Regular': '#3498db',
               'Light/Savers': '#2ecc71', 'Dormant': '#95a5a6'}

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for i, col in enumerate(plot_cols):
    ax = axes[i//3][i%3]
    for seg, color in seg_palette.items():
        subset = df[df['true_segment'] == seg][col]
        ax.hist(subset, bins=40, alpha=0.5, color=color, label=seg, density=True)
    ax.set_title(col, fontweight='bold', fontsize=10)
    ax.set_xlabel('Value')
    ax.set_ylabel('Density')
    if i == 0:
        ax.legend(fontsize=8)

plt.suptitle('Feature Distributions by True Segment', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### EDA 2: Pairplot of Key Features


In [ ]:
key_cols = ['monthly_transactions', 'avg_transaction_ksh', 'days_since_last_txn',
            'merchant_payment_ratio', 'savings_ratio', 'true_segment']
sample = df[key_cols].sample(600, random_state=42)
g = sns.pairplot(sample, hue='true_segment', plot_kws={'alpha': 0.4, 's': 15})
g.fig.suptitle('Pairplot of Mobile Money Features by True Segment', y=1.02, fontweight='bold')
plt.show()

### EDA 3: Transaction Volume vs Frequency


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for seg, color in seg_palette.items():
    mask = df['true_segment'] == seg
    axes[0].scatter(df.loc[mask, 'monthly_transactions'],
                    df.loc[mask, 'avg_transaction_ksh'],
                    alpha=0.3, s=15, color=color, label=seg)
axes[0].set_xlabel('Monthly Transactions')
axes[0].set_ylabel('Avg Transaction (KSH)')
axes[0].set_title('Transactions vs Avg Amount by Segment', fontweight='bold')
axes[0].legend()
axes[0].set_yscale('log')

df.boxplot(column='days_since_last_txn', by='true_segment', ax=axes[1])
axes[1].set_title('Days Since Last Transaction by Segment', fontweight='bold')
axes[1].set_xlabel('Segment')
axes[1].set_ylabel('Days')
axes[1].tick_params(axis='x', rotation=20)
plt.sca(axes[1])
plt.title('Recency by Segment')

plt.suptitle('Transaction Behavior Exploration', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## Preprocessing


In [ ]:
cluster_features = ['monthly_transactions', 'avg_transaction_ksh', 'account_age_months',
                    'merchant_payment_ratio', 'days_since_last_txn', 'savings_ratio',
                    'monthly_topup_count', 'unique_recipients_pm', 'bill_payments_pm']
X = df[cluster_features].values
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print(f"Scaled feature matrix: {X_scaled.shape}")

## Optimal K Selection: Elbow, Silhouette, Davies-Bouldin, Calinski-Harabasz


In [ ]:
K = range(2, 11)
inertias, sil_scores, db_scores, ch_scores = [], [], [], []

for k in K:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X_scaled, labels))
    db_scores.append(davies_bouldin_score(X_scaled, labels))
    ch_scores.append(calinski_harabasz_score(X_scaled, labels))

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes[0,0].plot(K, inertias, 'b-o', lw=2)
axes[0,0].axvline(x=4, color='red', ls='--', label='K=4')
axes[0,0].set_title('Elbow Method (Inertia)')
axes[0,0].set_xlabel('K'); axes[0,0].legend()

axes[0,1].plot(K, sil_scores, 'g-o', lw=2)
axes[0,1].axvline(x=4, color='red', ls='--', label='K=4')
axes[0,1].set_title('Silhouette Score (higher=better)')
axes[0,1].set_xlabel('K'); axes[0,1].legend()

axes[1,0].plot(K, db_scores, 'r-o', lw=2)
axes[1,0].axvline(x=4, color='green', ls='--', label='K=4')
axes[1,0].set_title('Davies-Bouldin (lower=better)')
axes[1,0].set_xlabel('K'); axes[1,0].legend()

axes[1,1].plot(K, ch_scores, 'm-o', lw=2)
axes[1,1].axvline(x=4, color='red', ls='--', label='K=4')
axes[1,1].set_title('Calinski-Harabasz (higher=better)')
axes[1,1].set_xlabel('K'); axes[1,1].legend()

plt.suptitle('Optimal K Selection Metrics', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("Metric summary:")
for i, k in enumerate(K):
    print(f"  K={k}: Silhouette={sil_scores[i]:.4f}, DB={db_scores[i]:.4f}, CH={ch_scores[i]:.1f}")

## Fit K-Means with K=4


In [ ]:
kmeans = KMeans(n_clusters=4, random_state=42, n_init=20)
df['cluster'] = kmeans.fit_predict(X_scaled)

cluster_profile = df.groupby('cluster')[cluster_features].mean()
print("Cluster Profiles:")
print(cluster_profile.round(2).to_string())
print(f"\nCluster sizes:\n{df['cluster'].value_counts().sort_index()}")
print(f"Final silhouette score: {silhouette_score(X_scaled, df['cluster']):.4f}")

# Assign names based on behavior
txn_means = df.groupby('cluster')['monthly_transactions'].mean()
days_means = df.groupby('cluster')['days_since_last_txn'].mean()
cluster_names_map = {}
for c in range(4):
    if txn_means[c] == txn_means.max(): cluster_names_map[c] = 'Heavy/Business'
    elif days_means[c] == days_means.max(): cluster_names_map[c] = 'Dormant'
remaining = [c for c in range(4) if c not in cluster_names_map]
remaining_sorted = sorted(remaining, key=lambda x: txn_means[x], reverse=True)
cluster_names_map[remaining_sorted[0]] = 'Regular'
cluster_names_map[remaining_sorted[1]] = 'Light/Savers'
df['cluster_name'] = df['cluster'].map(cluster_names_map)
print(f"\nCluster mapping: {cluster_names_map}")

## PCA 2D Visualization of Clusters + Profile Heatmap


In [ ]:
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
palette = {'Heavy/Business': '#e74c3c', 'Regular': '#3498db',
           'Light/Savers': '#2ecc71', 'Dormant': '#95a5a6'}
for name in df['cluster_name'].unique():
    mask = df['cluster_name'] == name
    axes[0].scatter(X_pca[mask, 0], X_pca[mask, 1], label=name,
                    alpha=0.5, s=15, color=palette.get(name, 'gray'))
centers_pca = pca.transform(kmeans.cluster_centers_)
axes[0].scatter(centers_pca[:, 0], centers_pca[:, 1], c='black', marker='X',
                s=250, zorder=5, label='Centroids')
axes[0].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
axes[0].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
axes[0].set_title('K-Means Clusters (PCA 2D)', fontweight='bold')
axes[0].legend(fontsize=9)

# Cluster profile heatmap
norm = MinMaxScaler().fit_transform(cluster_profile)
cluster_labels_ordered = [cluster_names_map[i] for i in range(4)]
sns.heatmap(pd.DataFrame(norm, index=cluster_labels_ordered, columns=cluster_features),
            ax=axes[1], cmap='YlOrRd', annot=True, fmt='.2f', linewidths=0.4)
axes[1].set_title('Cluster Profile Heatmap (Normalized)', fontweight='bold')
axes[1].tick_params(axis='x', rotation=40)
plt.tight_layout()
plt.show()

## Box Plots by Cluster


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
plot_feats = ['monthly_transactions', 'avg_transaction_ksh', 'days_since_last_txn',
              'savings_ratio', 'merchant_payment_ratio', 'monthly_volume_ksh']
for i, feat in enumerate(plot_feats):
    df.boxplot(column=feat, by='cluster_name', ax=axes[i//3][i%3])
    axes[i//3][i%3].set_title(feat, fontsize=10)
    axes[i//3][i%3].set_xlabel('')
    axes[i//3][i%3].tick_params(axis='x', rotation=20)
plt.suptitle('Feature Distribution by Cluster', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## Hierarchical Clustering Dendrogram


In [ ]:
sample_idx = np.random.choice(len(X_scaled), 150, replace=False)
Z = linkage(X_scaled[sample_idx], method='ward')
plt.figure(figsize=(14, 5))
dendrogram(Z, truncate_mode='level', p=4, leaf_rotation=90)
plt.title('Hierarchical Clustering Dendrogram (Ward Linkage, 150 users)', fontweight='bold')
plt.xlabel('User Sample')
plt.ylabel('Distance')
plt.tight_layout()
plt.show()

## Agglomerative Clustering Comparison


In [ ]:
agg = AgglomerativeClustering(n_clusters=4, linkage='ward')
agg_labels = agg.fit_predict(X_scaled)

sil_agg = silhouette_score(X_scaled, agg_labels)
sil_km  = silhouette_score(X_scaled, df['cluster'])

print(f"K-Means Silhouette:         {sil_km:.4f}")
print(f"Agglomerative Silhouette:   {sil_agg:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for c in range(4):
    mask = df['cluster'] == c
    axes[0].scatter(X_pca[mask, 0], X_pca[mask, 1], alpha=0.4, s=12, label=f'C{c}')
axes[0].set_title(f'K-Means (Sil={sil_km:.3f})', fontweight='bold')
axes[0].legend()

for c in range(4):
    mask = agg_labels == c
    axes[1].scatter(X_pca[mask, 0], X_pca[mask, 1], alpha=0.4, s=12, label=f'C{c}')
axes[1].set_title(f'Agglomerative Ward (Sil={sil_agg:.3f})', fontweight='bold')
axes[1].legend()

for ax in axes:
    ax.set_xlabel('PC1'); ax.set_ylabel('PC2')
plt.suptitle('K-Means vs Agglomerative Clustering', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## Cluster-Segment Agreement Analysis


In [ ]:
crosstab = pd.crosstab(df['cluster_name'], df['true_segment'],
                        normalize='index').round(3) * 100
print("Cluster vs True Segment (% of cluster):")
print(crosstab.round(1).to_string())

plt.figure(figsize=(10, 6))
sns.heatmap(crosstab, annot=True, fmt='.1f', cmap='Blues', linewidths=0.5)
plt.title('Cluster vs True Segment Overlap (%)', fontweight='bold')
plt.ylabel('K-Means Cluster')
plt.xlabel('True Segment')
plt.tight_layout()
plt.show()

## Cluster Radar Chart (Profile Summary)


In [ ]:
radar_features = ['monthly_transactions', 'avg_transaction_ksh', 'merchant_payment_ratio',
                  'savings_ratio', 'unique_recipients_pm', 'days_since_last_txn']
radar_labels = [f.replace('_', '\n') for f in radar_features]

normalized = MinMaxScaler().fit_transform(df.groupby('cluster_name')[radar_features].mean())
norm_df = pd.DataFrame(normalized,
                        index=df.groupby('cluster_name')[radar_features].mean().index,
                        columns=radar_features)

N = len(radar_features)
angles = np.linspace(0, 2*np.pi, N, endpoint=False).tolist()
angles += angles[:1]

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
colors_r = ['#e74c3c', '#3498db', '#2ecc71', '#95a5a6']
for i, (seg, row) in enumerate(norm_df.iterrows()):
    vals = row.values.tolist() + [row.values[0]]
    ax.plot(angles, vals, 'o-', lw=2, color=colors_r[i], label=seg)
    ax.fill(angles, vals, alpha=0.10, color=colors_r[i])

ax.set_xticks(angles[:-1])
ax.set_xticklabels(radar_labels, size=9)
ax.set_title('Cluster Radar Chart (Normalized Features)', fontweight='bold', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))
plt.tight_layout()
plt.show()

## Revenue & Engagement Summary by Cluster


In [ ]:
summary = df.groupby('cluster_name').agg(
    n_users=('cluster', 'count'),
    avg_monthly_vol_ksh=('monthly_volume_ksh', 'mean'),
    avg_transactions=('monthly_transactions', 'mean'),
    avg_days_inactive=('days_since_last_txn', 'mean'),
    avg_savings_ratio=('savings_ratio', 'mean'),
    avg_merchant_ratio=('merchant_payment_ratio', 'mean')
).round(2)
print("Cluster Business Summary:")
print(summary.to_string())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
summary['avg_monthly_vol_ksh'].plot(kind='bar', ax=axes[0], color=['#e74c3c','#3498db','#2ecc71','#95a5a6'],
                                     edgecolor='white', alpha=0.85)
axes[0].set_title('Avg Monthly Volume (KSH) by Cluster', fontweight='bold')
axes[0].set_xlabel('Cluster'); axes[0].set_ylabel('KSH')
axes[0].tick_params(axis='x', rotation=20)

summary['n_users'].plot(kind='bar', ax=axes[1], color=['#e74c3c','#3498db','#2ecc71','#95a5a6'],
                         edgecolor='white', alpha=0.85)
axes[1].set_title('Number of Users per Cluster', fontweight='bold')
axes[1].set_xlabel('Cluster'); axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=20)
plt.suptitle('Cluster Business Metrics', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## Conclusions

### Segment Descriptions

| Segment | Size | Key Traits | Monthly Volume |
|---------|------|-----------|---------------|
| **Heavy/Business** | ~400 | High transactions, high merchant ratio, frequent top-ups | Very High |
| **Regular** | ~700 | Moderate frequency, mixed use cases, active | Medium |
| **Light/Savers** | ~600 | Infrequent transactions, high savings ratio, low volume | Low |
| **Dormant** | ~300 | Rarely transact, long since last use, old accounts | Very Low |

### Revenue Potential

- **Heavy/Business** users generate disproportionate revenue through transaction fees, float income, and merchant commissions. Priority for retention and premium services.
- **Regular** users are the stable backbone — consistent fee income and growth candidates.

### Product Recommendations by Segment

**Heavy/Business:**
- Offer **business accounts** with bulk payment APIs
- **Working capital loans** (M-Shwari style) with instant approval
- **Loyalty rewards** — tiered cashback on transaction fees
- Priority customer service and dedicated relationship managers

**Regular:**
- **Salary advance** products (e.g., KCB M-Pesa)
- **Family plans** for peer-to-peer transfers
- Nudge toward merchant payments for everyday transactions

**Light/Savers:**
- **Savings goals** with interest incentives (lock-in savings products)
- **ROSCA/chama digitization** to move group savings online
- Affordable insurance products (crop, health)

**Dormant:**
- **Re-engagement campaigns**: bonus airtime, fee waivers for first transaction
- Survey to understand barriers (network, fees, trust)
- Simplified USSD onboarding for low-literacy users

### Clustering Performance
- K=4 confirmed by all four metrics (Elbow, Silhouette, Davies-Bouldin, Calinski-Harabasz)
- K-Means achieved **Silhouette > 0.45**, indicating well-separated clusters
- Agglomerative clustering with Ward linkage produces similar results, validating robustness
- High overlap between discovered clusters and true segments confirms meaningful behavioral patterns
